# Colab for DiffusionGemma sampling using HD

Please note that the codebase powering this demo prioritizes clear, modifiable code to encourage experimentation and development. As a result, it has not been optimized for inference speed.

In [ ]:
!pip install -q gemma

In [ ]:
# @title Imports

# for checkpoint
# Common imports
import os
from gemma import gm
from gemma import diffusion

# for sampling
import time
import jax
import jax.numpy as jnp
import dialog

from gemma.gm import text as gm_text
from hackable_diffusion import hd

from gemma.diffusion import _models as gemma_diffusion
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_network
from gemma.diffusion.hackable_diffusion_adapter.hd import hd_gemma_ar_state_handler

In [ ]:
# Initialize the Diffusion Gemma model
model = diffusion.DiffusionGemma_A26B_A4B()

# Load pretrained parameters
gemma_params = gm.ckpts.load_params(diffusion.CheckpointPath.DIFFUSIONGEMMA_A26B_A4B_IT, restore_concurrent_gb=16)
print("Checkpoint loaded.")

## HD Sampler

In [ ]:
# @title Padding utility functions

def format_prompt(
    prompt: str,
    format_str: dialog.Format,
    system_instruction: str | None = None,
    add_think_token: bool = False,
) -> str:
  """Formats a prompt for Gemma."""
  transformations = []
  system_transforms = []
  if add_think_token:
    system_transforms.append(dialog.Think())
  if system_instruction is not None:
    system_transforms.append(system_instruction)
  if system_transforms:
    transformations.append(dialog.System(*system_transforms))
  transformations.append(dialog.User(prompt))
  conv = dialog.Conversation(*transformations)
  # See third_party/py/gemma/gm/text/_chat_sampler.py;l=322
  return conv.as_text(format=format_str, add_tool_response_tag_after_call=True)


def tokenize_prompts_right_pad_fixed_max_length(
    tokenizer,
    prompts,
    system_instruction: str | None = None,
    add_think_token: bool = False,
    pad_token: int = 0,
    max_length: int = 64,
) -> tuple[jnp.ndarray, jnp.ndarray]:
  """Tokenizes and right-pads a batch of prompts to equal length.

  Matches Gemma convention: real tokens first, padding at the end.

  Args:
    tokenizer: Tokenizer with a ``FORMAT`` attribute and ``encode`` method.
    prompts: Batch of text prompts.
    system_instruction: System instruction to prepend to the prompt.
    add_think_token: Whether to add a think token to the system instruction.
    pad_token: Token id used for padding.

  Returns:
    A tuple of:
      - Padded token array of shape ``[B, max_prompt_len]`` (right-padded).
      - Array of per-element prompt lengths of shape ``[B]``.
  """
  if not prompts:
    raise ValueError("Must have at least one prompt.")

  all_token_ids = []
  for prompt in prompts:
    formatted = format_prompt(
        prompt,
        format_str=tokenizer.FORMAT,
        system_instruction=system_instruction,
        add_think_token=add_think_token,
    )
    prompt_ids = tokenizer.encode(formatted, add_bos=True)
    all_token_ids.append(prompt_ids)

  prompt_lengths = jnp.array(
      [len(ids) for ids in all_token_ids], dtype=jnp.int32
  )
  # max_len = max(len(ids) for ids in all_token_ids)

  # Right-pad: real tokens first, pad tokens at the end.
  padded = []
  for ids in all_token_ids:
    if len(ids) < max_length:
      padded.append(ids + [pad_token] * (max_length - len(ids)))
    else:
      raise ValueError(f"Prompt length {len(ids)} exceeds max length {max_length}.")

  return jnp.array(padded, dtype=jnp.int32), prompt_lengths

### Trajectory Sampling

In [ ]:
# @title  Define canvas sampler

# 1. Load Model & Tokenizer
gemma_model = gemma_diffusion.DiffusionGemma_A26B_A4B()

gemma_tokenizer = gm_text.Gemma4Tokenizer()
print("Tokenizer ready.")

# 2. Define Sampling Constants
CANVAS_LENGTH = 512        #@param {type:"integer"}
MAX_DENOISING_STEPS = 20   #@param {type:"integer"}
MAX_PROMPT_LENGTH = 64     #@param {type:"integer"}
# 3. Setup Hackable Diffusion Process & Stepper
diffusion_process = hd.corruption.CategoricalProcess.uniform_process(
    schedule=hd.corruption.LinearDiscreteSchedule(),
    num_categories=gemma_tokenizer.vocab_size,
)

stop_tokens = (
    gemma_tokenizer.special_tokens.EOS,
    gemma_tokenizer.special_tokens.END_OF_TURN,
    gemma_tokenizer.special_tokens.BEGIN_OF_TOOL_RESPONSE,
)



# Define stepper
stepper = hd.sampling.DiscreteDDIMStep(
    corruption_process=diffusion_process,
    temperature=0.7,
    logits_dtype=jnp.bfloat16,
)

canvas_sampler = hd.sampling.DiffusionSampler(
    time_schedule=hd.sampling.UniformTimeSchedule(),
    stepper=stepper,
    num_steps=MAX_DENOISING_STEPS,
    update_conditioning_fn=hd_gemma_ar_state_handler.PropagateSelfConditioningFn(),
    store_trajectory=True
)

# 5. Define JITted Sampling Function
# @jax.jit
def sample_fn(params, rng_key, p_tokens, p_lengths):
  """JIT compiled autoregressive sampling function."""
  # Move instantiation inside so JAX doesn't try to hash them at the boundary
  diffusion_network = hd_gemma_network.WrappedDiffusionGemmaNetwork(
      gemma_model=gemma_model
  )
  inference_fn = hd.inference.FlaxLinenInferenceFn(
      network=diffusion_network,
      params={'gemma_model': params},
  )
  gemma_state_handler = hd_gemma_ar_state_handler.GemmaARStateHandler(
    gemma_network=diffusion_network,
    gemma_params=params,
    end_tokens=stop_tokens,
  )

  sampler_state = gemma_state_handler.init_ar_state(
      batch_size=len(p_tokens),
      conditioning={
          "prompt_tokens": p_tokens,
          "prompt_lengths": p_lengths,
      },
      canvas_length=CANVAS_LENGTH,
      max_num_canvases=1,
  )
  # Propagate random number generator.
  canvas_init_rng, canvas_sampler_rng = jax.random.split(rng_key)

  # Create new canvas.
  initial_canvas = diffusion_process.sample_from_invariant(
    key=canvas_init_rng,
    data_spec=jnp.zeros((len(p_tokens), CANVAS_LENGTH, 1), dtype=jnp.int32),
  )
  # Sample canvas via diffusion.
  return canvas_sampler(
    inference_fn=inference_fn,
    rng=canvas_sampler_rng,
    initial_noise=initial_canvas,
    conditioning=gemma_state_handler.create_conditioning_from_state(
        sampler_state=sampler_state
    ),
  )

class HDDiffusionSampler:
  def __init__(self, params, tokenizer, max_length):
    self.params = params
    self.tokenizer = tokenizer
    self.max_length = max_length

  def sample(self, prompt, *, rng=None):
    if isinstance(prompt, str):
      prompt = [prompt]
    if rng is None:
      rng = jax.random.PRNGKey(42)

    # Tokenize
    prompt_tokens, prompt_lengths = tokenize_prompts_right_pad_fixed_max_length(
        self.tokenizer, prompt
    )

    # Cast to int32 before padding to ensure JAX cache hit
    prompt_tokens = jnp.array(prompt_tokens, dtype=jnp.int32)
    prompt_lengths = jnp.array(prompt_lengths, dtype=jnp.int32)

    # Execute
    return sample_fn(self.params, rng, prompt_tokens, prompt_lengths)

sampler = HDDiffusionSampler(
    params=gemma_params,
    tokenizer=gemma_tokenizer,
    max_length=MAX_PROMPT_LENGTH
)

PROMPT = "Summarize why text diffusion models are better than AR models" # @param {type:"string"}
rng = jax.random.PRNGKey(42)
print(f"User: {PROMPT}")
print("\nSampling (This might take 2-3 minutes)...")
tic = time.time()
last_step, trajectory = sampler.sample(PROMPT, rng=rng)
print(f"\nSampling takes: {time.time()-tic:.1f}s.")
# === Extract results to CPU immediately ===
final_tokens = np.array(last_step.xt[..., 0])           # numpy, on CPU
trajectory_np = np.array(trajectory.xt[:, :, :, 0])  # (steps, batch, seq_len)
trajectory_tokens = [trajectory_np[i, 0, :].tolist() for i in range(trajectory_np.shape[0])] # numpy, on CPU
text = gemma_tokenizer.decode(final_tokens[0].tolist())
print('\nText Output:\n-------------------------------------\n', text)
# === Free GPU memory ===
del last_step, trajectory

In [ ]:
# @title Render GIF for trajectory visualization

from PIL import Image as PILImage, ImageDraw, ImageFont
import numpy as np
import random

GIBBERISH = list('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789!@#$%&*?+=<>~^')

def make_diffusion_gif(
    trajectory_tokens,
    tokenizer,
    output_path='diffusion_trajectory_scan.gif',
    prompt=PROMPT,
    font_size=28,
    hold_ms=400,
    chars_per_line=80,
    margin=30,
    num_scan_frames=12,
    scan_ms=40,
    glow_radius=12,
    band_width=80,
    last_frame_ms=5000,
):
  font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", font_size)
  title_font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", font_size + 8)

  tmp = PILImage.new('RGB', (1, 1))
  tmp_draw = ImageDraw.Draw(tmp)
  cell_w = tmp_draw.textbbox((0, 0), 'M', font=font)[2] + 2
  cell_h = int(tmp_draw.textbbox((0, 0), 'Mg', font=font)[3] * 1.5)

  def safe_decode(tok_id, use_whitespace=False):
      text = tokenizer.decode([int(tok_id)])
      text = text.replace('\t', '→')
      out = []
      for ch in text:
          if ch == '\n':
              out.append('\n')
          elif 32 <= ord(ch) < 127:
              out.append(ch)
          else:
              out.append(' ' if use_whitespace else random.choice(GIBBERISH))
      return ''.join(out) or '·'

  num_steps = len(trajectory_tokens)
  num_tokens = len(trajectory_tokens[0])
  final_tokens = trajectory_tokens[-1]

  all_decoded = []
  for s in range(num_steps):
      step_decoded = []
      for tok_idx in range(num_tokens):
          matches_final = trajectory_tokens[s][tok_idx] == final_tokens[tok_idx]
          step_decoded.append(safe_decode(
              trajectory_tokens[s][tok_idx],
              use_whitespace=matches_final,
          ))
      all_decoded.append(step_decoded)
  print("Tokens decoded.")

  # Final text defines the board
  final_decoded = all_decoded[-1]
  cells_per_token = [len(final_decoded[tok_idx]) for tok_idx in range(num_tokens)]
  total_chars = sum(cells_per_token)

  def fit_to_board(step_idx):
      result = []
      for tok_idx in range(num_tokens):
          allocated = cells_per_token[tok_idx]
          decoded = all_decoded[step_idx][tok_idx]
          if len(decoded) <= allocated:
              padded = decoded.ljust(allocated)
          else:
              padded = decoded[:allocated]
          for ch in padded:
              result.append((ch, tok_idx))
      return result

  GREEN = (80, 220, 100)
  RED = (255, 80, 80)
  GREY = (130, 130, 140)
  BG = (25, 25, 35)
  WHITE = (220, 220, 235)
  GLOW_COLOR = np.array([80, 180, 255], dtype=np.float32)

  # Pre-compute all boards
  all_boards = [fit_to_board(s) for s in range(num_steps)]

  # Compute grid positions respecting \n in final text
  final_board = all_boards[-1]
  positions = []
  col = 0
  row = 0
  for i, (ch, tok_idx) in enumerate(final_board):
      if ch == '\n':
          positions.append(None)
          col = 0
          row += 1
      else:
          positions.append((col, row))
          col += 1
          if col >= chars_per_line:
              col = 0
              row += 1

  actual_num_lines = row + 1
  header_h = cell_h * (4 if prompt else 3)
  img_w = margin * 2 + chars_per_line * cell_w
  img_h = header_h + actual_num_lines * cell_h + margin

  print(f"Board: {total_chars} cells, {actual_num_lines} lines")

  def compute_colors(step_idx, prev_tokens, prev_colors):
      cur_tokens = trajectory_tokens[step_idx]
      colors = []
      for tok_idx in range(num_tokens):
          changed = prev_tokens is not None and cur_tokens[tok_idx] != prev_tokens[tok_idx]
          matches_final = cur_tokens[tok_idx] == final_tokens[tok_idx]
          if matches_final:
              colors.append('green')
          elif changed:
              colors.append('red')
          elif prev_colors and prev_colors[tok_idx] == 'green':
              colors.append('green')
          else:
              colors.append('grey')
      return colors

  def render_full_frame(board, tok_colors, step_idx):
      img = PILImage.new('RGB', (img_w, img_h), color=BG)
      draw = ImageDraw.Draw(img)

      if prompt:
          draw.text((margin, margin), f'Prompt: {prompt}',
                    fill=(180, 180, 200), font=font)

      title_y = margin + cell_h if prompt else margin
      draw.text((margin, title_y), f'Denoising Step {step_idx}/{num_steps - 1}',
                fill=WHITE, font=title_font)

      n_green = sum(1 for c in tok_colors if c == 'green')
      pct = int(n_green / num_tokens * 100)
      draw.text((margin + 500, title_y), f'{pct}% converged',
                fill=(100, 100, 120), font=font)

      bar_y = title_y + cell_h + 10
      bar_w = img_w - margin * 2
      draw.rounded_rectangle([margin, bar_y, margin + bar_w, bar_y + 10],
                             radius=5, fill=(60, 60, 70))
      progress = step_idx / max(num_steps - 1, 1)
      if progress > 0:
          draw.rounded_rectangle([margin, bar_y, margin + int(bar_w * progress), bar_y + 10],
                                 radius=5, fill=(100, 180, 255))

      color_map = {'green': GREEN, 'red': RED, 'grey': GREY}
      for i, (ch, tok_idx) in enumerate(board):
          if positions[i] is None:
              continue
          col_idx, row_idx = positions[i]
          x = margin + col_idx * cell_w
          y = header_h + row_idx * cell_h
          color = color_map[tok_colors[tok_idx]] if 0 <= tok_idx < len(tok_colors) else GREY
          if ch.strip():
              cw = draw.textbbox((0, 0), ch, font=font)[2]
              x_offset = (cell_w - cw) // 2
              draw.text((x + x_offset, y), ch, fill=color, font=font)

      return img

  # Pre-compute vertical gradient lookup
  ys = np.arange(img_h, dtype=np.float32)

  def make_scanline_frame(old_arr, new_arr, scan_y):
      alpha = np.clip((scan_y - ys) / band_width + 0.5, 0.0, 1.0)
      alpha = alpha * alpha * (3 - 2 * alpha)
      alpha = alpha.reshape(-1, 1, 1)

      blended = new_arr * alpha + old_arr * (1 - alpha)

      for offset in range(-2, 3):
          y = scan_y + offset
          if 0 <= y < img_h:
              intensity = 1.0 - abs(offset) / 3
              blended[y, :] = np.clip(
                  blended[y, :] + GLOW_COLOR * intensity * 0.4, 0, 255
              )

      return PILImage.fromarray(blended.astype(np.uint8))

  # Compute all colors
  all_colors = []
  prev_tokens = None
  prev_colors = None
  for s in range(num_steps):
      colors = compute_colors(s, prev_tokens, prev_colors)
      all_colors.append(colors)
      prev_tokens = trajectory_tokens[s]
      prev_colors = colors

  # Render static frames
  step_frames = []
  for step_idx in range(num_steps):
      print(f'\rRendering frame {step_idx+1}/{num_steps}', end='')
      step_frames.append(render_full_frame(all_boards[step_idx], all_colors[step_idx], step_idx))
  print(' Done!')

  # Convert to numpy once
  step_arrays = [np.array(f, dtype=np.float32) for f in step_frames]

  # Build animation with scanline transitions
  frames = []
  durations = []
  print("Building transitions...")

  for step_idx in range(num_steps):
      print(f'\rStep {step_idx+1}/{num_steps}', end='')

      if step_idx > 0:
          has_changes = any(
              trajectory_tokens[step_idx][i] != trajectory_tokens[step_idx - 1][i]
              or all_colors[step_idx][i] != all_colors[step_idx - 1][i]
              for i in range(num_tokens)
          )

          if has_changes:
              old_arr = step_arrays[step_idx - 1]
              new_arr = step_arrays[step_idx]
              for f in range(num_scan_frames):
                  scan_y = int(header_h + (img_h - header_h) * f / num_scan_frames)
                  frames.append(make_scanline_frame(old_arr, new_arr, scan_y))
                  durations.append(scan_ms)

      frames.append(step_frames[step_idx])
      durations.append(hold_ms)

  durations[-1] = last_frame_ms
  print(' Done!')

  frames[0].save(output_path, save_all=True, append_images=frames[1:],
                 duration=durations, loop=0)
  print(f"Saved to {output_path} ({len(frames)} frames from {num_steps} steps)")


make_diffusion_gif(trajectory_tokens, gemma_tokenizer, prompt=PROMPT)

from IPython.display import display, Image
display(Image(filename='diffusion_trajectory_scan.gif'))

### Full AR Sampling

In [ ]:
# @title Define HD Sampler and sampling constants

# 1. Load Model & Tokenizer
gemma_model = gemma_diffusion.DiffusionGemma_A26B_A4B()

gemma_tokenizer = gm_text.Gemma4Tokenizer()
print("Tokenizer ready.")

# 2. Define Sampling Constants
CANVAS_LENGTH = 128        #@param {type:"integer"}
MAX_DENOISING_STEPS = 20   #@param {type:"integer"}
MAX_NUM_CANVASES = 8       #@param {type:"integer"}
MAX_PROMPT_LENGTH = 64     #@param {type:"integer"}

# 3. Decide the components
stepper_type = "DiscreteDDIM" #@param ["DiscreteDDIM", "IntegratedDiscreteDDIM", "DiscreteFlowMatching"]
routing_strategy = "None" # @param ["None", "Greedy"]
use_early_stopping = True # @param {type:"boolean"}

# 3. Setup Hackable Diffusion Process & Stepper
diffusion_process = hd.corruption.CategoricalProcess.uniform_process(
    schedule=hd.corruption.LinearDiscreteSchedule(),
    num_categories=gemma_tokenizer.vocab_size,
)

stop_tokens = (
    gemma_tokenizer.special_tokens.EOS,
    gemma_tokenizer.special_tokens.END_OF_TURN,
    gemma_tokenizer.special_tokens.BEGIN_OF_TOOL_RESPONSE,
)

# Define early stopping strategy
if use_early_stopping:
  early_stop_fn = hd.sampling.DiffusionEntropyEarlyStopFn()
else:
  early_stop_fn = hd.sampling.DiffusionNoEarlyStopFn()


# Define routing strategy
match routing_strategy:
  case "None":
    planner = None
  case "Greedy":
    planner = hd.sampling.GreedyPlanner()


# Define stepper
match stepper_type:
  case "DiscreteDDIM":
    stepper = hd.sampling.DiscreteDDIMStep(
        corruption_process=diffusion_process,
        planner=planner,
        temperature=0.7,
        logits_dtype=jnp.bfloat16,
    )
  case "IntegratedDiscreteDDIM":
    stepper = hd.sampling.IntegratedDiscreteDDIMStep(
        corruption_process=diffusion_process,
        temperature=0.7,
        logits_dtype=jnp.bfloat16,
    )
  case "DiscreteFlowMatching":
    stepper = hd.sampling.DiscreteFlowMatchingStep(
        corruption_process=diffusion_process,
        planner=planner,
        temperature=0.7,
        logits_dtype=jnp.bfloat16,
    )


# 4. Create Canvas Sampler
canvas_sampler = hd.sampling.DiffusionSamplerWithEarlyStopping(
    time_schedule=hd.sampling.UniformTimeSchedule(),
    stepper=stepper,
    num_steps=MAX_DENOISING_STEPS,
    update_conditioning_fn=hd_gemma_ar_state_handler.PropagateSelfConditioningFn(),
    early_stopping_fn=hd.sampling.DiffusionEntropyEarlyStopFn(),
)

# 5. Define JITted Sampling Function
@jax.jit
def sample_ar_fn(params, rng_key, p_tokens, p_lengths):
  """JIT compiled autoregressive sampling function."""
  # Move instantiation inside so JAX doesn't try to hash them at the boundary
  diffusion_network = hd_gemma_network.WrappedDiffusionGemmaNetwork(
      gemma_model=gemma_model
  )
  inference_fn = hd.inference.FlaxLinenInferenceFn(
      network=diffusion_network,
      params={'gemma_model': params},
  )
  gemma_state_handler = hd_gemma_ar_state_handler.GemmaARStateHandler(
    gemma_network=diffusion_network,
    gemma_params=params,
    end_tokens=stop_tokens,
  )
  autoregressive_sampler = hd.sampling.AutoregressiveDiffusionSampler(
      diffusion_process=diffusion_process,
      canvas_sampler=canvas_sampler,
      max_num_canvases=MAX_NUM_CANVASES,
      canvas_length=CANVAS_LENGTH,
      state_handler=gemma_state_handler,
      data_dtype=jnp.int32,
      data_shape=(1,),
  )

  return autoregressive_sampler(
      diffusion_inference_fn=inference_fn,
      batch_size=len(p_tokens),
      rng=rng_key,
      conditioning={
          "prompt_tokens": p_tokens,
          "prompt_lengths": p_lengths,
      },
  )

class HDDiffusionSampler:
  def __init__(self, params, tokenizer, max_length):
    self.params = params
    self.tokenizer = tokenizer
    self.max_length = max_length

  def sample(self, prompt, *, rng=None):
    if isinstance(prompt, str):
      prompt = [prompt]
    if rng is None:
      rng = jax.random.PRNGKey(42)

    # Tokenize
    prompt_tokens, prompt_lengths = tokenize_prompts_right_pad_fixed_max_length(
        self.tokenizer, prompt
    )

    # Cast to int32 before padding to ensure JAX cache hit
    prompt_tokens = jnp.array(prompt_tokens, dtype=jnp.int32)
    prompt_lengths = jnp.array(prompt_lengths, dtype=jnp.int32)

    # Execute
    return sample_ar_fn(self.params, rng, prompt_tokens, prompt_lengths)

sampler = HDDiffusionSampler(
    params=gemma_params,
    tokenizer=gemma_tokenizer,
    max_length=MAX_PROMPT_LENGTH
)

rng = jax.random.PRNGKey(42)
tic = time.time()
ar_diffusion_samples = sampler.sample("Hello", rng=rng)
print(f"\nSampling takes: {time.time()-tic:.1f}s.")
out_text_ar = gemma_tokenizer.decode(ar_diffusion_samples[0][0])
del ar_diffusion_samples
print('\nText Output:\n-------------------------------------\n', out_text_ar)
print("HD Sampler ready.")

In [ ]:
import numpy as np

# @title Sampling #1
PROMPT = "What is the meaning of life and the rest?" # @param {type:"string"}

print(f"\nSampling: {PROMPT!r}")
rng = jax.random.PRNGKey(42)
tic = time.time()
ar_diffusion_samples = sampler.sample(PROMPT, rng=rng)
print(f"\nSampling takes: {time.time()-tic:.1f}s.")
print("\n--- Autoregressive Multi-Canvas Sampling Result ---")
out_text_ar = gemma_tokenizer.decode(ar_diffusion_samples[0][0])
del ar_diffusion_samples
print('\nText Output:\n-------------------------------------\n', out_text_ar)

In [ ]:
# @title Sampling #2
PROMPT = "What are the advantages of text diffusion models over AR models?" # @param {type:"string"}

print(f"\nSampling: {PROMPT!r}")
rng, _ = jax.random.split(rng)
tic = time.time()
ar_diffusion_samples = sampler.sample(PROMPT, rng=rng)
print(f"\nSampling takes: {time.time()-tic:.1f}s.")
print("\n--- Autoregressive Multi-Canvas Sampling Result ---")
out_text_ar = gemma_tokenizer.decode(ar_diffusion_samples[0][0])
del ar_diffusion_samples
print('\nText Output:\n-------------------------------------\n', out_text_ar)